# 07_rag_chain

对应 `index.md` 第 7 阶段：检索 + 组 RAG 链。

笔记本在 `codes/langchain/07_rag_chain/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT` 并加载 `codes/langchain/local.env`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

本阶段把前六阶段拼成一条链：**检索 → 拼上下文 → 填提示 → 调模型**。真实 ChatModel 仍用 `init_chat_model("deepseek:...")`，密钥 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_MODEL`）。

检索侧继续用 `DeterministicFakeEmbedding`——DeepSeek 没有 Embedding API。**假向量不按语义命中**，所以检索出来的文档常看着不着调（问「猫」可能命中「狗」）。这是预期的，不影响链路本身，别当成 bug 去改。

In [1]:
import os
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "07_rag_chain" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "07_rag_chain"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

def load_local_env(env_path: Path) -> None:
    """把 local.env 写入 os.environ（已有键不覆盖）。"""
    if not env_path.is_file():
        return
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("\"'")
        if key and key not in os.environ:
            os.environ[key] = value

ROOT = lab_root()
load_local_env(ROOT.parent / "local.env")
ROOT, bool(os.environ.get("DEEPSEEK_API_KEY"))

(PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langchain/07_rag_chain'),
 True)

## 1. 检索出上下文

RAG 的第一半是检索。已有 `store`。

用 `as_retriever(search_kwargs={"k": 2})` 得到 `retriever`，对问题 `"猫喜欢吃什么？"` 调 `retriever.invoke`。

打印命中条数；再逐条打印 `page_content` 与 `metadata["source"]`。

In [2]:
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.vectorstores import InMemoryVectorStore

emb = DeterministicFakeEmbedding(size=32)
docs = [
    Document(page_content="猫喜欢吃鱼。", metadata={"source": "a.txt"}),
    Document(page_content="狗喜欢追球。", metadata={"source": "b.txt"}),
    Document(page_content="向量库用来做相似度检索。", metadata={"source": "c.txt"}),
]
store = InMemoryVectorStore.from_documents(docs, emb)

# 作答

retriver = store.as_retriever(search_kwargs={"k": 2})

retrived = retriver.invoke("猫喜欢吃什么？")

for doc in retrived:
    print(doc.page_content)
    print(doc.metadata.get("source", "(missing)"))
#评阅
# 对。as_retriever(k=2)、invoke、逐条打印 page_content 与 source 都做了；.get() 带兜底是好习惯。
# 缺一半：题面还要「打印命中条数」，没打 len(retrived)。
# 另外变量名 retriver 拼错了（retriever）；前后一致所以能跑，但建议改掉。

#参考答案
# retriever = store.as_retriever(search_kwargs={"k": 2})
# hits = retriever.invoke("猫喜欢吃什么？")
# print(len(hits))
# for doc in hits:
#     print(doc.page_content, doc.metadata["source"])


向量库用来做相似度检索。
c.txt
狗喜欢追球。
b.txt


## 2. 把检索结果拼成上下文

模型只会读到一个字符串，不会读 `Document` 列表，所以中间要有一步拼接。

已有 `docs`（`Document` 列表）。写一个函数 `format_docs(docs)`：把各条 `page_content` 用两个换行 `"\n\n"` 连接成一个字符串并返回。

打印 `format_docs(docs)` 的结果，以及参与拼接的文档条数。

In [5]:
from langchain_core.documents import Document

docs = [
    Document(page_content="猫喜欢吃鱼。", metadata={"source": "a.txt"}),
    Document(page_content="狗喜欢追球。", metadata={"source": "b.txt"}),
]

# 作答

def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(d.page_content for d in docs)

print(format_docs(docs))
print(len(docs))
#评阅
# 对。join 写法、双换行分隔、条数都对了；类型标注也合理。

#参考答案
# def format_docs(docs):
#     return "\n\n".join(d.page_content for d in docs)
# print(format_docs(docs))
# print(len(docs))


猫喜欢吃鱼。

狗喜欢追球。
2


## 3. 建一个带 `{context}` 的提示

用 `ChatPromptTemplate.from_messages` 建模板，两条消息：

- system：正文里含占位符 `{context}`，并写清「只根据上下文回答，上下文没提到就说不知道」
- human：正文里含占位符 `{question}`

用 `format_messages(context="猫喜欢吃鱼。", question="猫喜欢吃什么？")` 打印格式化后的消息列表，确认两个占位符都被填上了。

In [8]:
from langchain_core.prompts import ChatPromptTemplate

# 作答

prompt = ChatPromptTemplate.from_messages([
    ("system", "根据{context}回答, 上下文没提到就说不知道"),
    ("human", "{question}")
])

ms = prompt.format_messages(context="猫喜欢吃鱼。", question="猫喜欢吃什么？")

for m in ms:
    print(m.content)
#评阅
# 对。两条消息、两个占位符都填上了，消息数也是 2。
# 两处可改：
# 1) system 漏了「只」——「根据上下文回答」和「只根据上下文回答」是两个意思，后者才是约束。
# 2) {context} 直接嵌在句子里（「根据猫喜欢吃鱼。回答」），上下文与指令混成一句。
#    生产里会把资料夹在分隔符/标签里，让模型分得清哪段是指令、哪段是材料。

#参考答案
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "只根据下面的上下文回答，上下文没提到就说不知道。\n\n上下文：\n{context}"),
#     ("human", "{question}"),
# ])
# for m in prompt.format_messages(context="猫喜欢吃鱼。", question="猫喜欢吃什么？"):
#     print(m.content)


根据猫喜欢吃鱼。回答, 上下文没提到就说不知道
猫喜欢吃什么？


## 4. 手动跑一遍 RAG

已有 `retriever`、`prompt`、`model`、`format_docs`。`question` 取 `"猫喜欢吃什么？"`。

按四步走，中间结果各打印一次：

1. `retriever.invoke(question)` 得到 `docs`
2. 拼成 `context`（可复用 `format_docs`）
3. `prompt.format_messages(context=context, question=question)` 得到消息
4. 喂给 `model.invoke`，打印 `response.content`

这是 RAG 的骨架：先检索材料，再夹进提示，最后交给模型。

前置代码的 `retriever` 取 `k=3`，把三条文档一次取满——假向量不按语义排序，取满才能保证上下文里真的有答案，免得你以为链写错了。

In [11]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore

emb = DeterministicFakeEmbedding(size=32)
docs = [
    Document(page_content="猫喜欢吃饕餮。", metadata={"source": "a.txt"}),
    Document(page_content="狗喜欢追球。", metadata={"source": "b.txt"}),
    Document(page_content="向量库用来做相似度检索。", metadata={"source": "c.txt"}),
]
store = InMemoryVectorStore.from_documents(docs, emb)
retriever = store.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([
    ("system", "只根据下面的上下文回答，上下文没提到就说不知道。\n\n上下文：\n{context}"),
    ("human", "{question}"),
])

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)

def format_docs(docs):
    """第 2 题的写法。"""
    return "\n\n".join(d.page_content for d in docs)

question = "猫喜欢吃什么？"

# 作答

docs = retriever.invoke(question)

context = format_docs(docs)

msgs = prompt.format_messages(context=context, question=question)

response = model.invoke(msgs)

print(response.content)
#评阅
# 对。四步齐全，中间结果（docs / context / msgs）都有，最后打 response.content。
# 注意：前置代码里那条「猫喜欢吃鱼。」被你改成了「猫喜欢吃饕餮。」，所以模型答「猫喜欢吃饕餮。」。
#   这反倒把 RAG 的本质演示清楚了——模型就是在照上下文说话。
#   不过约定是前置代码不用改，改了会和后面几题对不上。

#参考答案
# docs = retriever.invoke(question)
# context = format_docs(docs)
# msgs = prompt.format_messages(context=context, question=question)
# print(model.invoke(msgs).content)


猫喜欢吃饕餮。


## 5. 用 LCEL 把链串起来

第 4 题那四步手写太长了。用 `|` 组一条链。

中间有一环容易踩：提示模板要的是**带键的字典**（`{"context": ..., "question": ...}`），不是单个字符串。所以不能把 `retriever | format_docs` 直接接到 `prompt` 上（那样 `prompt` 会收到一个 `str`，报 `Expected mapping type`）。要先给「上下文」和「原问题」各接一条分支、凑成字典，再喂给提示；原问题用 `RunnablePassthrough()` 原样传过去。

链的形状：**（检索拼上下文 ∥ 原问题）→ 字典 → 提示 → 模型 → 取纯文本**。

对 `"猫喜欢吃什么？"` 调 `invoke`，打印返回的字符串。注意返回值应是 `str`，不是 `AIMessage`。

In [13]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore

emb = DeterministicFakeEmbedding(size=32)
docs = [
    Document(page_content="猫喜欢吃鱼。", metadata={"source": "a.txt"}),
    Document(page_content="狗喜欢追球。", metadata={"source": "b.txt"}),
    Document(page_content="向量库用来做相似度检索。", metadata={"source": "c.txt"}),
]
store = InMemoryVectorStore.from_documents(docs, emb)
retriever = store.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([
    ("system", "只根据下面的上下文回答，上下文没提到就说不知道。\n\n上下文：\n{context}"),
    ("human", "{question}"),
])

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)

def format_docs(docs):
    """第 2 题的写法。"""
    return "\n\n".join(d.page_content for d in docs)

# 作答

chain = (
    retriever | format_docs |
    RunnableLambda(lambda d: {"context": d, "question": "猫喜欢吃什么？"}) |
    prompt |
    model |
    StrOutputParser()
)

response = chain.invoke("猫喜欢吃什么？")

print(response)
#评阅
# 链跑通了，返回的确实是纯字符串，但这里有个真问题：问题被写死在 RunnableLambda 里。
# chain.invoke("...") 传进去的入参被丢掉了，实测：
#   chain.invoke("猫喜欢吃什么？") -> 猫喜欢吃鱼。
#   chain.invoke("狗喜欢玩什么？") -> 猫喜欢吃鱼。   <-- 入参失效
# 这样链只能回答这一个问题，失去了复用意义；RunnablePassthrough 的意义正是把原问题原样透传。
# 另外写成字典那一步不必套 RunnableLambda，直接给 dict 即可，LCEL 会自动并行分发。

#参考答案
# chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | prompt
#     | model
#     | StrOutputParser()
# )
# print(chain.invoke("猫喜欢吃什么？"))


猫喜欢吃鱼。


## 6. 让链同时交出答案和来源

只给答案不够——用户要能看答案依据哪几段。已有 `retriever`、`prompt`、`model`、`format_docs`。

用 `RunnableParallel` 组一条链，使 `invoke("猫喜欢吃什么？")` 返回一个字典，含两个键：

- `"answer"`：模型基于上下文写的答案（字符串）
- `"sources"`：本次命中的 `metadata["source"]` 列表

打印 `answer`；再打印 `sources`。

注意 `RunnableParallel` 会把**同一个输入**分发给每条分支，所以答案分支得自己再拿一次检索结果。

In [18]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore

emb = DeterministicFakeEmbedding(size=32)
docs = [
    Document(page_content="猫喜欢吃鱼。", metadata={"source": "a.txt"}),
    Document(page_content="狗喜欢追球。", metadata={"source": "b.txt"}),
    Document(page_content="向量库用来做相似度检索。", metadata={"source": "c.txt"}),
]
store = InMemoryVectorStore.from_documents(docs, emb)
retriever = store.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([
    ("system", "只根据下面的上下文回答，上下文没提到就说不知道。\n\n上下文：\n{context}"),
    ("human", "{question}"),
])

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)

def format_docs(docs):
    """第 2 题的写法。"""
    return "\n\n".join(d.page_content for d in docs)

# 作答

from langchain_core.runnables import RunnableLambda

chain = RunnableParallel(
    answer=(
        retriever
        | format_docs
        | (lambda d: {"context": d, "question": "猫喜欢吃什么？"})
        | prompt
        | model
        | StrOutputParser()
    ),
    sources=(
        retriever
        | (lambda docs: [doc.metadata["source"] for doc in docs])
    )
)

response = chain.invoke("猫喜欢吃什么？")

print(response)
#评阅
# 对。RunnableParallel 用法、两个键、sources 取 metadata["source"] 都对了。
# 与第 5 题同一个问题：answer 分支把问题写死了，实测 invoke("狗喜欢玩什么？") 仍返回「猫喜欢吃鱼。」。
#   改透传后 RunnablePassthrough 才用得上（当前那条 import 是空挂着）。
# 另外题面要「打印 answer；再打印 sources」，这里直接 print 整个字典，能看到但不算分开打。

#参考答案
# chain = RunnableParallel(
#     answer=(
#         {"context": retriever | format_docs, "question": RunnablePassthrough()}
#         | prompt | model | StrOutputParser()
#     ),
#     sources=retriever | (lambda ds: [d.metadata["source"] for d in ds]),
# )
# res = chain.invoke("猫喜欢吃什么？")
# print(res["answer"])
# print(res["sources"])


{'answer': '猫喜欢吃鱼。', 'sources': ['c.txt', 'b.txt', 'a.txt']}
